In [ ]:
# BETA TUNING

from simulator import run_pytorch_monte_carlo
from util import create_final_matrix
from helpers import (
    calculate_market_rmse,
    safe_scrape_elo,
    country_to_elo,
)
import numpy as np
from constants import TEAM_ID_MAP, ALPHA
import pandas as pd

elo_df = safe_scrape_elo()


# format: [Home, Away, Home_Elo, Away_Elo, Bookie_Odds(H, D, A)]
validation_set = [
    (
        "Switzerland",
        "Colombia",
        country_to_elo(elo_df, "Switzerland"),
        country_to_elo(elo_df, "Colombia"),
        [3.90, 3.19, 2.48],
    ),
    (
        "Argentina",
        "Egypt",
        country_to_elo(elo_df, "Argentina"),
        country_to_elo(elo_df, "Egypt"),
        [1.38, 5.07, 13.00],
    ),
    (
        "USA",
        "Belgium",
        country_to_elo(elo_df, "USA"),
        country_to_elo(elo_df, "Belgium"),
        [2.62, 3.50, 2.98],
    ),
    (
        "Portugal",
        "Spain",
        country_to_elo(elo_df, "Portugal"),
        country_to_elo(elo_df, "Spain"),
        [4.12, 3.67, 2.06],
    ),
    (
        "Mexico",
        "England",
        country_to_elo(elo_df, "Mexico"),
        country_to_elo(elo_df, "England"),
        [3.23, 3.37, 2.60],
    ),
    (
        "Brazil",
        "Norway",
        country_to_elo(elo_df, "Brazil"),
        country_to_elo(elo_df, "Norway"),
        [1.85, 3.86, 5.07],
    ),
]

beta_candidates = np.linspace(start=0.0001, stop=0.0005, num=12)

best_beta = None
lowest_error = 999.0

global_q_matrix = pd.read_csv("./data/global_q_matrix.csv")
global_q_grid = pd.read_csv("./data/global_q_grid.csv")

print("[*] Beginning Beta Grid Search...")

for beta in beta_candidates:
    total_rmse = 0.0
    for match in validation_set:
        home, away, elo_h, elo_a, odds = match
        test_q_grid = create_final_matrix(
            home,
            TEAM_ID_MAP[home],
            away,
            TEAM_ID_MAP[away],
            elo_h,
            elo_a,
            global_q_matrix,
            ALPHA,
            beta,
        )

        p_h, p_d, p_a = run_pytorch_monte_carlo(test_q_grid, num_simulations=5000)
        model_probs = [p_h, p_d, p_a]
        error = calculate_market_rmse(model_probs, odds)
        total_rmse += error

    avg_rmse = total_rmse / len(validation_set)
    print(f"Beta: {beta:.6f} | Average Market RMSE: {avg_rmse:.4f}")

    if avg_rmse < lowest_error:
        best_beta = beta
        lowest_error = avg_rmse
        print(f"New best beta found: {best_beta}!")
print(f"\n[+] Optimal Beta Found: {best_beta:.6f} (Error: {lowest_error:.4f})")

In [ ]:
# OPTIMISED 2D GRID SEARCH WITH PRE-CACHING
from simulator import run_pytorch_monte_carlo
from helpers import calculate_market_rmse, safe_scrape_elo, country_to_elo
from util import (
    neutralise_global_prior,
    standardise_possessions,
    align_team_perspective,
    create_full_team_df,
    calculate_global_q,
)
import numpy as np
from constants import TEAM_ID_MAP
import pandas as pd
from tqdm import tqdm
from blueprint import MatrixPipeline, EloModifier

elo_df = safe_scrape_elo()

validation_matchups = [
    ("Switzerland", "Colombia", [3.90, 3.19, 2.48]),
    ("Argentina", "Egypt", [1.38, 5.07, 13.00]),
    ("USA", "Belgium", [2.62, 3.50, 2.98]),
    ("Portugal", "Spain", [4.12, 3.67, 2.06]),
    ("Mexico", "England", [3.23, 3.37, 2.60]),
    ("Brazil", "Norway", [1.85, 3.86, 5.07]),
]

alpha_candidates = np.linspace(start=0.01, stop=0.6, num=20)  # Expanded to 20
beta_candidates = np.linspace(start=0.0001, stop=0.0006, num=20)  # Expanded to 20
total_iterations = len(alpha_candidates) * len(beta_candidates)

global_q_matrix = pd.read_csv("./data/global_q_matrix.csv")

# =====================================================================
# STEP 1: PRE-CACHE ALL STATIC PANDAS OPERATIONS (Run once on CPU)
# =====================================================================
print("[*] Pre-caching historical transition counts for validation matchups...")
cached_validation_set = []
neutral_prior = neutralise_global_prior(global_q_matrix)  # Pre-symmetrize once

for home, away, bookie_odds in validation_matchups:
    # Build core context
    elo_home = country_to_elo(elo_df, home)
    elo_away = country_to_elo(elo_df, away)

    # Process historical data once
    full_home_df = standardise_possessions(create_full_team_df(home))
    full_away_df = standardise_possessions(create_full_team_df(away))

    aligned_home_df = align_team_perspective(
        full_home_df, TEAM_ID_MAP[home], sim_role="H"
    )
    aligned_away_df = align_team_perspective(
        full_away_df, TEAM_ID_MAP[away], sim_role="A"
    )

    home_counts, _ = calculate_global_q(aligned_home_df)
    away_counts, _ = calculate_global_q(aligned_away_df)

    cached_validation_set.append(
        {
            "home_team": home,
            "away_team": away,
            "home_id": TEAM_ID_MAP[home],
            "away_id": TEAM_ID_MAP[away],
            "elo_home": elo_home,
            "elo_away": elo_away,
            "bookie_odds": bookie_odds,
            "home_counts": home_counts,
            "away_counts": away_counts,
        }
    )

# =====================================================================
# STEP 2: LIGHTWEIGHT HYPERPARAMETER LOOP (Pure Math & GPU Sims)
# =====================================================================
best_alpha, best_beta, lowest_error = None, None, 999.0
baseline_pipeline = MatrixPipeline([EloModifier()])

print(f"[*] Beginning fast 2D Grid Search ({total_iterations} combos)...")
with tqdm(
    total=total_iterations, desc="Optimising Hyperparameters", unit="grid"
) as pbar:
    for alpha in alpha_candidates:
        for beta in beta_candidates:
            total_rmse = 0.0

            for match in cached_validation_set:
                # Inject active loop hyperparameters into context
                ctx = match.copy()
                ctx["alpha"] = alpha
                ctx["beta"] = beta

                # Modified build_grid_fast accepts the pre-cached counts directly
                test_q_grid = baseline_pipeline.build_grid_fast(neutral_prior, ctx)

                # PyTorch GPU execution
                p_h, p_d, p_a = run_pytorch_monte_carlo(
                    test_q_grid, num_simulations=3000, verbose=False
                )

                error = calculate_market_rmse([p_h, p_d, p_a], match["bookie_odds"])
                total_rmse += error

            avg_rmse = total_rmse / len(cached_validation_set)

            if avg_rmse < lowest_error:
                tqdm.write(
                    f"[!] New best found! Alpha: {alpha:.4f} | Beta: {beta:.6f} | RMSE: {avg_rmse:.4f}"
                )
                lowest_error = avg_rmse
                best_alpha = alpha
                best_beta = beta
            pbar.update(1)

[*] Pre-caching historical transition counts for validation matchups...
[*] Found 4 matches for Switzerland. Parsing...
[+] Successfully built dataframe for Switzerland (5067 events).
[*] Found 4 matches for Colombia. Parsing...
[+] Successfully built dataframe for Colombia (4918 events).
[*] Found 4 matches for Argentina. Parsing...
[+] Successfully built dataframe for Argentina (6002 events).
[*] Found 4 matches for Egypt. Parsing...
[+] Successfully built dataframe for Egypt (5715 events).
[*] Found 4 matches for USA. Parsing...
[+] Successfully built dataframe for USA (5103 events).
[*] Found 4 matches for Belgium. Parsing...
[+] Successfully built dataframe for Belgium (5693 events).
[*] Found 4 matches for Portugal. Parsing...
[+] Successfully built dataframe for Portugal (5323 events).
[*] Found 4 matches for Spain. Parsing...
[+] Successfully built dataframe for Spain (5546 events).
[*] Found 4 matches for Mexico. Parsing...
[+] Successfully built dataframe for Mexico (4671 eve

Optimising Hyperparameters:   0%|          | 1/400 [00:06<41:58,  6.31s/grid]

[!] New best found! Alpha: 0.0100 | Beta: 0.000100 | RMSE: 0.1148


Optimising Hyperparameters:   1%|          | 3/400 [00:18<40:52,  6.18s/grid]

[!] New best found! Alpha: 0.0100 | Beta: 0.000153 | RMSE: 0.1113


...

Optimising Hyperparameters:  27%|██▋       | 108/400 [11:20<31:00,  6.37s/grid]

[!] New best found! Alpha: 0.1653 | Beta: 0.000284 | RMSE: 0.0801


Optimising Hyperparameters:  32%|███▏      | 128/400 [13:25<28:03,  6.19s/grid]

[!] New best found! Alpha: 0.1963 | Beta: 0.000284 | RMSE: 0.0787


Optimising Hyperparameters: 100%|██████████| 400/400 [44:17<00:00,  6.64s/grid]
